# Homework 11 APPM 4600
## Problem 1: Composite Quadrature Rules
### Part (a)
Construct a composite quadrature rules using degree-2 interpolation with open nodes.

We will first consider a single panel $[x_j, x_{j+1}]$ of width $h=x_{j+1} - x_j$. Since we are using an open rule with degree 2, use 3 interior nodes. 
$$X_0 = x_j + \frac 1 4 h,\hspace{.25cm} X_1 = x_j + \frac 1 2 h, \hspace{.25cm} X_2 = x_j + \frac 3 4 h.$$
And map the panel to $[0,1]$ to use the orthonormal Lagrange polynomials. Define
$$t = \frac{x-x_j}{h}$$
so the nodes become
$$t_0 = \frac 1 4, \hspace{.25cm} t_1 = \frac 1 2, \hspace{.25cm} t_2 = \frac 3 4.$$
And approximate the integral
$$ \int_{x_j}^{x_{j+1}} f(x) dx = h \int_0^1 f(x_j + ht)dt. $$

Now we construct the Lagrange polynomial through the nodes
$$p(t) = \sum_{i=0}^2 f(X_i)\ell_i (t)$$
so the quadrature rule is
$$\int_0^1 f(x_j + ht) dt \approx \sum_{i=0}^n f(X_i) \int_0^1 \ell_i (t) dt$$
with weights
$$w_i = h \int_0^1 \ell_i (t) dt.$$

After carrying out the integration,
$$\int_{x_j}^{x_{j+1}} f(x) dx \approx h\left[ \frac 2 3 f(x_j + \frac 1 4 h) - \frac 1 3 f(x_j + \frac 1 2 h) + \frac 2 3 f(x_j + \frac 3 4 h) \right]$$

Note: the middle weight is negative! Not good for stability.

Now, we divide $[a,b]$ into $N$ panels of width $h = \frac{b-a}{N}$, so the full rule becomes
$$I_{o2} = \sum_{j=0}^{N-1} h\left[ \frac 2 3 f(x_j + \frac 1 4 h) - \frac 1 3 f(x_j + \frac 1 2 h) + \frac 2 3 f(x_j + \frac 3 4 h) \right]$$

### Part (b)
Compute an upper bound on the error.

For a degree 2 interpolating rule, the error per panel is 
$$E_{panel} = Ch^4 f^{(3)}(\xi_j)$$
Then there are $N = \frac{b-a}{h}$ panels, so
$$E_{global} \sim N \cdot h^4 = \frac{b-a}{h} \cdot h^4 = (b-a)h^3$$
giving
$$|E_{global}| \leq C(b-a)h^3 \max_{x\in [a,b]}|f^{(3)}(x)|$$

So we get $O(h^3)$ convergence.


### Part (c)
Compare this with Simpson

Simpson is the better choice here, unless we have singularities at the endpoints. Simpson includes the endpoints, has $O(h^4)$ convergence, and has better stability (our method has negative weights leading to possible subtractive canclation). However, our method would be better if the endpoints of our domain had singularities.

## Problem 2: Gauss-Laguerre Quadrature
### Part (a)
We want
$$I = \int_{-\infty}^{\infty} \frac{1}{1+x^2} dx = 2 \int_0^\infty \frac{1}{1+x^2}dx$$
but define $g(x) = f(x)e^x = \frac{e^x}{1+x^2}$, so
$$\int_0^\infty f(x) dx \approx \sum_{i=1}^n w_i g(x_i)$$

In [43]:
# Compute the nodes and weights for Gauss-Laguerre quadrature and use these to 
# esimate I. How small can you make the error, and which n is this?

# Note that f(x) = f(x)e^x e^-x, so define g(x) = f(x)e^x = e^x/(1+x^2)

import numpy as np
from scipy.special import roots_laguerre

def runge(x):
    return 1/(1 + x**2)

def compute_integral_a(n):
    x, w = roots_laguerre(n)  # nodes and weights

    g = np.exp(x) * runge(x)
    I_half = np.sum(w * g)

    return 2 * I_half  # symmetry

# test convergence
true_val = np.pi

for n in [5, 10, 20, 40, 80, 120]:
    approx = compute_integral_a(n)
    err = abs(approx - true_val)
    print(n, approx, err)

5 3.018980148398545 0.12261250519124811
10 3.0851526838469177 0.05643996974287546
20 3.1144787277869947 0.027113925802798367
40 3.128390115857257 0.01320253773253599
80 3.1351120983655902 0.0064805552242028774
120 3.1373064627356326 0.004286190854160488


The error decreases very slowly every iteration. I'm not sure it would be feasible to reach machine precision.

### Part (b)
Perform change of variables to $x = e^t -1$. Then $dx = e^t dt$ so
$$\int_0^\infty \frac{1}{1+x^2} dx = \int_0^\infty \frac{e^t}{1+(e^t -1)^2} dt$$
and rewrite in the Leguerre form
$$\int_0^\infty f(t) dt = \int_0^\infty f(t) e^t e^{-t} dt$$
so define
$$g(t) = f(t)e^t = \frac{e^{2t}}{1+(e^t -1)^2}.$$

In [44]:
def transformed_integrand(t):
    return np.exp(2*t) / (1 + (np.exp(t) - 1)**2)

def compute_integral_b(n):
    x, w = roots_laguerre(n)

    g = transformed_integrand(x)
    I_half = np.sum(w * g)

    return 2 * I_half

# test convergence
for n in [5, 10, 20, 40, 80, 120]:
    approx = compute_integral_b(n)
    err = abs(approx - np.pi)
    print(n, approx, err)

5 3.0501818244775953 0.09141082911219778
10 3.1519756432913333 0.010382989701540168
20 3.1402645306485315 0.0013281229412616113
40 3.141567389580248 2.52640095452783e-05
80 3.1415925853993976 6.819039555239215e-08
120 nan nan


/var/folders/40/26f5s8qj42z2mxw1gfrnh5rm0000gn/T/ipykernel_89022/247556722.py:2: RuntimeWarning: overflow encountered in exp
  return np.exp(2*t) / (1 + (np.exp(t) - 1)**2)
/var/folders/40/26f5s8qj42z2mxw1gfrnh5rm0000gn/T/ipykernel_89022/247556722.py:2: RuntimeWarning: overflow encountered in square
  return np.exp(2*t) / (1 + (np.exp(t) - 1)**2)
/var/folders/40/26f5s8qj42z2mxw1gfrnh5rm0000gn/T/ipykernel_89022/247556722.py:2: RuntimeWarning: invalid value encountered in divide
  return np.exp(2*t) / (1 + (np.exp(t) - 1)**2)


We see much better convergence here. We are quickly in well outpacing the $n=120$ iteration first method with only $n=40$ iterations here.

### Part (c)
Explain...

The Gauss-Laguerre method assumes the integrand behaves like $f(x)e^{-x}$, but we forced $g(x) = \frac{e^x}{1+x^2}$, which grows exponentially, causing catastrophic cancllation and poor quadrature accurayc. 

Part (b) works better since the change of variables reshapes the integrand to be $\frac{e^{2t}}{1+(e^t-1)^2}$, which behaves more smoothly. 

## Problem 3: Romberg Integration
We want to compute 
$$\int_0^{50} \cos(x) dx = sin(50)$$
With the composite midpoint rule, we have
$$M_n = h \sum f(x_i + \frac h 2)$$

In [53]:
def f(x):
    return np.cos(x)

# composite midpoint rule
def midpoint_rule(a, b, n):
    h = (b - a) / n
    midpoints = a + (np.arange(n) + 0.5) * h
    return h * np.sum(f(midpoints))

# Romberg using midpoint rule
def romberg_midpoint(a, b, max_k):
    # R[k][j] table
    R = np.zeros((max_k, max_k))
    
    for k in range(max_k):
        n = 2**k
        R[k, 0] = midpoint_rule(a, b, n)
        
        for j in range(1, k+1):
            R[k, j] = R[k, j-1] + (R[k, j-1] - R[k-1, j-1]) / (4**j - 1)
    
    return R

# run it
a, b = 0, 50
max_k = 13   # up to n = 4096
R = romberg_midpoint(a, b, max_k)

true_val = np.sin(50)

print("\nRomberg Table (errors)\n")

max_cols = R.shape[0]

for k in range(max_cols):
    n = 2**k
    row = []
    for j in range(k+1):
        err = abs(R[k, j] - true_val)
        row.append(f"{err:10.3e}")  # scientific notation
    
    print(f"n={n:<5} | " + "  ".join(row))


Romberg Table (errors)

n=1     |  4.982e+01
n=2     |  4.971e+01   4.968e+01
n=4     |  4.969e+01   4.968e+01   4.968e+01
n=8     |  4.915e+01   8.210e+01   9.089e+01   9.312e+01
n=16    |  1.476e-01   1.619e+01   2.274e+01   2.454e+01   2.501e+01
n=32    |  2.872e-02   1.091e-02   1.068e+00   1.445e+00   1.547e+00   1.573e+00
n=64    |  6.793e-03   5.160e-04   1.766e-04   1.677e-02   2.250e-02   2.404e-02   2.443e-02
n=128   |  1.676e-03   3.030e-05   2.079e-06   6.907e-07   6.506e-05   8.711e-05   9.301e-05   9.450e-05
n=256   |  4.175e-04   1.865e-06   3.048e-08   2.031e-09   6.695e-10   6.292e-08   8.421e-08   8.989e-08   9.134e-08
n=512   |  1.043e-04   1.161e-07   4.690e-10   7.444e-12   4.922e-13   1.618e-13   1.520e-11   2.035e-11   2.172e-11   2.207e-11
n=1024  |  2.607e-05   7.252e-09   7.300e-12   2.881e-14   2.776e-16   2.220e-16   1.665e-16   1.110e-15   1.443e-15   1.499e-15   1.499e-15
n=2048  |  6.516e-06   4.532e-10   1.139e-13   2.220e-16   1.110e-16   1.110e-16   1

Although both rules are $O(h^2)$, trapezoidal rule's error expansion only contains even powers of $h$. This makes it better since each step cancels the leading error term. Further, trapezoidal method is better for refining from $n$ to $2n$ nodes, where we can reuse all previous function evaluations, only needing to calculate new midpoints. 